In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 3.9 Waveguides and Cavity Resonances

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume III — Classical Electrodynamics",
    number="3.9",
    title="Waveguides and Cavity Resonances",
    blurb="Light in a box: confining electromagnetic waves with conducting walls "
    "turns the wave equation into an eigenvalue problem, with discrete modes, cutoff "
    "frequencies, and resonances — the same mathematics as a vibrating drumhead.",
    difficulty="advanced",
    estimate="120–150 min",
)

## Notebook overview

The free plane wave of [§3.8](maxwell-waves.ipynb) could take any frequency and travel
in any direction. Put
it inside a hollow metal pipe and everything changes: the conducting walls force
boundary conditions on the field, and only a **discrete** set of transverse patterns,
the **modes**, can propagate, each with its own **cutoff frequency** below which it
simply will not go. Close the pipe at both ends and even the frequencies become
discrete: the box rings only at its **resonances**. Confinement quantizes.

This notebook is a convergence of three threads the course has been building. The wave
equation restricted to a cross-section is a **Helmholtz eigenvalue problem**; solving
it is exactly the boundary-value problem of [§3.4](laplace-poisson.ipynb) (the
discretized Laplacian), read as the eigenvalue problem of
[§0.5](../00-foundations/eigenvalues-svd.ipynb), giving the normal modes of
[§2.7](../02-classical-mechanics/small-oscillations.ipynb), now for electromagnetic
fields. We solve it twice: analytically for the separable rectangular guide, where the
modes are products of sines and cosines, and **numerically** with a sparse eigensolver
on the discretized cross-section, which also cracks shapes no separation of variables
can touch.

The running example is the standard **WR-90 X-band guide** ($a=22.9\,$mm,
$b=10.2\,$mm). We find its mode patterns and cutoffs, watch a mode switch from
evanescent to propagating as the frequency crosses cutoff, recover the cutoffs from a
sparse eigenproblem, take that solver to a non-separable cross-section, and finally
close the guide into a 3-D cavity with discrete resonances. Throughout runs one
parallel: the rectangular-guide eigenproblem is the *same* mathematics as a vibrating
drumhead, and the cavity the same as a 3-D box, the classical rehearsal for the
quantum particle in a box (Vol VI).

Everything is in **SI units**, with $c=1/\sqrt{\mu_0\varepsilon_0}$. A guided wave does
propagate, so exactly one figure here is animated, the dominant mode travelling down
the guide; everything else is a still.

> **How to read the checks.** Each exercise ends with a `validate` call against an
> independent fact: a sine mode satisfying the transverse Helmholtz equation, the
> dominant cutoff equal to $c/2a$, a sparse eigensolver recovering the analytic
> cutoffs, a cavity resonance matching the box formula. A ✓ is strong evidence; a ✗ is
> a prompt to *locate the discrepancy*, not a verdict.
>
> **A resolution note.** The sparse cross-section eigensolver recovers the **low**-order
> cutoffs to about 0.1%, but the **high**-order modes drift downward: the five-point
> Dirichlet Laplacian under-estimates eigenvalues, and a fixed grid under-resolves
> their finer structure (too few nodes per half-wavelength). We
> validate on the well-resolved low modes and read the drift as the resolution lesson
> it is, the same honesty applied to the [§3.4](laplace-poisson.ipynb) corner and the
> [§3.6](magnetostatics.ipynb) $\nabla\cdot\mathbf B$ residual.

> **Scope.** A working review, not a full course. See Nolting, *Theoretical Physics 3*
> {cite}`nolting3`; Griffiths, *Introduction to Electrodynamics* {cite}`griffiths_em`
> (ch. 9); Jackson {cite}`jackson` (ch. 8).

## Theory in brief

### Confining a wave

Inside a hollow perfect conductor the field still obeys the vacuum wave equation of
[§3.8](maxwell-waves.ipynb), but now with the boundary condition that the **tangential
electric field
vanishes** on the walls. Seeking solutions that propagate along the guide axis
$\hat{\mathbf z}$ and keep a fixed transverse shape,

```{math}
:label: eq-confined-wave
\mathbf E(x,y,z,t) = \mathbf E_T(x,y)\,e^{i(\beta z-\omega t)},
```

separates the problem: the transverse pattern $\psi(x,y)$ obeys the **Helmholtz
equation** $\nabla_T^2\psi+k_c^2\psi=0$ on the cross-section, with $k_c$ a separation
constant fixed by the boundary.

### TE and TM modes

The guided fields split into two families: **transverse electric** (TE, $E_z=0$) and
**transverse magnetic** (TM, $B_z=0$); Jackson, *Classical Electrodynamics*, ch. 8,
carries the reduction of Maxwell's equations to these two families out in full. For a
rectangular guide $a\times b$ the patterns
are products of sines and cosines; the TM modes, with $E_z=0$ on the walls, are

```{math}
:label: eq-te-tm
\psi_{mn}^{\mathrm{TM}} \propto \sin\frac{m\pi x}{a}\,\sin\frac{n\pi y}{b}, \qquad
k_c^2 = \Big(\frac{m\pi}{a}\Big)^2 + \Big(\frac{n\pi}{b}\Big)^2,
```

with integers $(m,n)$ labelling the mode (TE modes use cosines, the Neumann partner).

### Cutoff frequency

Substituting the travelling form {eq}`eq-confined-wave` into the wave equation ties
frequency and propagation constant to the separation constant,
$(\omega/c)^2=\beta^2+k_c^2$; each mode therefore has a **cutoff frequency**

```{math}
:label: eq-cutoff
f_c = \frac{c\,k_c}{2\pi}, \qquad \beta = \sqrt{(\omega/c)^2 - k_c^2}.
```

When $\omega/c>k_c$ the propagation constant $\beta$ is real and the mode travels with
guide wavelength $\lambda_g=2\pi/\beta>\lambda_{\mathrm{free}}$. When $\omega/c<k_c$,
$\beta$ is imaginary: the mode is **evanescent**, decaying as $e^{-\alpha z}$ with
$\alpha=\sqrt{k_c^2-(\omega/c)^2}$, and carries no power. The mode of lowest cutoff (the
**dominant** mode, TE$_{10}$ for a rectangular guide, $f_c=c/2a$) sets the single-mode
band over which a guide is normally used.

### The eigenvalue connection

The transverse Helmholtz problem **is** an eigenvalue problem for the
boundary-constrained Laplacian,

```{math}
:label: eq-eigenproblem
-\nabla_T^2\,\psi = k_c^2\,\psi,
```

its eigenvalues the squared cutoffs $k_c^2$ and its eigenvectors the mode patterns.
This unifies three earlier ideas: the discretized Laplacian of the
[§3.4](laplace-poisson.ipynb) boundary-value problem, the eigenvalue machinery of
[§0.5](../00-foundations/eigenvalues-svd.ipynb), and the normal modes of
[§2.7](../02-classical-mechanics/small-oscillations.ipynb). We solve it
analytically for the separable rectangle and numerically with a sparse eigensolver,
which then handles cross-sections that do not separate.

### Cavity resonances

Cap the guide at both ends ($z=0$ and $z=d$) and the field is confined in all three
directions; only a discrete set of standing-wave frequencies survives,

```{math}
:label: eq-cavity
f_{mnp} = \frac{c}{2}\sqrt{\Big(\frac{m}{a}\Big)^2+\Big(\frac{n}{b}\Big)^2+\Big(\frac{p}{d}\Big)^2}.
```

These are the resonances of a microwave cavity, and the same box eigenproblem that, in
Vol VI, sets the energy levels of a quantum particle in a box.

### The drumhead parallel

None of this is special to electromagnetism. The rectangular-guide eigenproblem is
mathematically identical to a vibrating rectangular **membrane** (a 2-D box Helmholtz
problem), and the cavity to a 3-D box. *Confinement produces discrete modes* is a
universal theme, the same one met for coupled oscillators in
[§2.7](../02-classical-mechanics/small-oscillations.ipynb) and waiting in the
quantum bound states of Vol VI.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import eigsh

from ecp import draw, validate
from ecp.animate import show

from scipy.constants import mu_0 as MU0  # vacuum permeability, T·m/A
from scipy.constants import epsilon_0 as EPS0  # vacuum permittivity, F/m

C_LIGHT = 1.0 / np.sqrt(MU0 * EPS0)  # speed of light, m/s
A_WR90, B_WR90 = 22.9e-3, 10.2e-3  # WR-90 X-band guide cross-section, m
ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT


def kc2_rect(m, n, a=A_WR90, b=B_WR90):
    """Squared cutoff wavenumber of a rectangular-guide mode (eq-te-tm).

    k_c^2 = (mπ/a)^2 + (nπ/b)^2, the separation constant set by the walls; its
    square root times c/2π is the mode's cutoff frequency.

    Parameters
    ----------
    m, n : int
        Mode indices along the wide (a) and narrow (b) walls.
    a, b : float, optional
        Guide cross-section dimensions in metres (default WR-90).

    Returns
    -------
    float
        The squared cutoff wavenumber k_c^2, in 1/m^2.
    """
    return (m * np.pi / a) ** 2 + (n * np.pi / b) ** 2


def cutoff_freq(m, n, a=A_WR90, b=B_WR90):
    """Cutoff frequency of a rectangular-guide mode (eq-cutoff).

    f_c = c·k_c/2π: below it the mode is evanescent, above it the mode
    propagates.

    Parameters
    ----------
    m, n : int
        Mode indices.
    a, b : float, optional
        Cross-section dimensions in metres (default WR-90).

    Returns
    -------
    float
        The cutoff frequency, in hertz.
    """
    return C_LIGHT * np.sqrt(kc2_rect(m, n, a, b)) / (2.0 * np.pi)


def tm_pattern(m, n, X, Y, a=A_WR90, b=B_WR90):
    """Longitudinal field E_z of a rectangular TM_mn mode (eq-te-tm).

    The sine product sin(mπx/a)·sin(nπy/b) that vanishes on the walls, the
    eigenfunction of the Dirichlet transverse Laplacian.

    Parameters
    ----------
    m, n : int
        Mode indices (both >= 1 for TM).
    X, Y : numpy.ndarray
        Coordinate grids on the cross-section.
    a, b : float, optional
        Cross-section dimensions in metres (default WR-90).

    Returns
    -------
    numpy.ndarray
        The E_z pattern on the grid (arbitrary amplitude).
    """
    return np.sin(m * np.pi * X / a) * np.sin(n * np.pi * Y / b)


def te_pattern(m, n, X, Y, a=A_WR90, b=B_WR90):
    """Longitudinal field H_z of a rectangular TE_mn mode (eq-te-tm).

    The cosine product cos(mπx/a)·cos(nπy/b) satisfying the Neumann (wall)
    condition; it shares the cutoff k_c^2 of its TM partner.

    Parameters
    ----------
    m, n : int
        Mode indices (not both zero).
    X, Y : numpy.ndarray
        Coordinate grids on the cross-section.
    a, b : float, optional
        Cross-section dimensions in metres (default WR-90).

    Returns
    -------
    numpy.ndarray
        The H_z pattern on the grid (arbitrary amplitude).
    """
    return np.cos(m * np.pi * X / a) * np.cos(n * np.pi * Y / b)


def neg_laplacian(mask, hx, hy):
    """Sparse negative transverse Laplacian with Dirichlet walls over a mask.

    Builds the 5-point finite-difference operator of §3.4 (with separate x and
    y spacings) on the ``True`` nodes of ``mask``; any neighbour outside the
    mask is held at zero (a perfect-conductor wall). Its smallest eigenvalues
    are the squared cutoffs k_c^2 (eq-eigenproblem). The mask lets the same
    builder serve a rectangle or a non-separable cross-section.

    Parameters
    ----------
    mask : numpy.ndarray of bool
        Interior nodes, indexed ``mask[j, i]`` with ``j`` over y and ``i`` over x.
    hx, hy : float
        Grid spacings along x and y, in metres.

    Returns
    -------
    A : scipy.sparse.csr_matrix
        The N × N operator over the N ``True`` nodes.
    index : numpy.ndarray
        Map from grid position to row index (``-1`` where outside the mask).
    """
    index = -np.ones(mask.shape, dtype=int)
    nodes = np.argwhere(mask)
    for n, (j, i) in enumerate(nodes):
        index[j, i] = n
    N = len(nodes)
    A = lil_matrix((N, N))
    for n, (j, i) in enumerate(nodes):
        A[n, n] = 2.0 / hx**2 + 2.0 / hy**2
        for dj, di, h in [(0, -1, hx), (0, 1, hx), (-1, 0, hy), (1, 0, hy)]:
            jj, ii = j + dj, i + di
            if 0 <= jj < mask.shape[0] and 0 <= ii < mask.shape[1] and mask[jj, ii]:
                A[n, index[jj, ii]] += -1.0 / h**2
    return A.tocsr(), index


def cavity_freq(m, n, p, a, b, d):
    """Resonant frequency of a rectangular cavity mode (eq-cavity).

    f_mnp = (c/2)·sqrt((m/a)^2 + (n/b)^2 + (p/d)^2), the standing-wave
    frequency of a box closed on all six sides.

    Parameters
    ----------
    m, n, p : int
        Mode indices along a, b, d.
    a, b, d : float
        Cavity dimensions in metres.

    Returns
    -------
    float
        The resonant frequency, in hertz.
    """
    return 0.5 * C_LIGHT * np.sqrt((m / a) ** 2 + (n / b) ** 2 + (p / d) ** 2)

## Exercise 1 — The rectangular waveguide and its modes

A hollow rectangular pipe of width $a$ and height $b$ is the workhorse of microwave
engineering. Inside it the transverse field of a TM$_{mn}$ mode is the sine product
{eq}`eq-te-tm`, $\psi\propto\sin(m\pi x/a)\sin(n\pi y/b)$, which vanishes on every wall
and so satisfies the perfect-conductor boundary condition, with squared cutoff
$k_c^2=(m\pi/a)^2+(n\pi/b)^2$ ({numref}`fig-wg-crosssection`). We use the standard
**WR-90 X-band guide**, $a=22.9\,$mm and $b=10.2\,$mm.

**This exercise (worked).**

1. Evaluate the mode patterns on the cross-section grid and confirm
   numerically that a sine mode satisfies the transverse Helmholtz equation
   $\nabla_T^2\psi=-k_c^2\psi$: apply the 5-point Laplacian (the
   [§3.4](laplace-poisson.ipynb) stencil) to the sampled $\psi$ and compare
   to $-k_c^2\psi$ on the
   interior, away from the wall nodes where the one-sided stencil is inexact.
2. Draw the transverse patterns of the lowest modes ({numref}`fig-wg-modes`)
   — the familiar half-wave humps of a constrained membrane.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.check(
    helm_err < 1e-2,
    "the sine-product mode satisfies the transverse Helmholtz equation ∇²ψ = −k_c²ψ",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 2 — Cutoff frequencies and the dominant mode

Each mode carries a cutoff frequency $f_c=c\,k_c/2\pi$ {eq}`eq-cutoff` below which it
cannot propagate. The mode with the lowest cutoff is the **dominant** mode; for a
rectangular guide with $a>b$ that is TE$_{10}$, whose cutoff is simply $f_c=c/2a$. The
band between the dominant cutoff and the next mode's is the **single-mode band**, where
only one pattern propagates and the guide is normally operated.

**This exercise (worked).** Compute $f_c$ for the low modes, confirm the dominant-mode
cutoff equals $c/2a$, and order the modes by cutoff to read off the single-mode band
({numref}`fig-wg-cutoffs`).

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    fc_TE10,
    C_LIGHT / (2 * A_WR90),
    "the dominant-mode cutoff is c/(2a) (TE10)",
    rtol=1e-6,
)
validate.check(
    order[0] == (1, 0) and order[1] == (2, 0),
    "the modes order by cutoff with TE10 dominant and TE20 next",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 3 — Evanescent versus propagating

Whether a mode travels depends entirely on how its drive frequency compares to its
cutoff. The guide propagation constant is $\beta=\sqrt{(\omega/c)^2-k_c^2}$
{eq}`eq-cutoff`. Above cutoff $\beta$ is real and the mode propagates with guide
wavelength $\lambda_g=2\pi/\beta$; below cutoff the quantity under the root is negative,
$\beta$ is imaginary, and the mode is **evanescent**, dying away as $e^{-\alpha z}$ with
$\alpha=\sqrt{k_c^2-(\omega/c)^2}$.

**This exercise (worked).** For the dominant TE$_{10}$ mode ($f_c=6.55\,$GHz), evaluate
$\beta$ at $6$, $8$, and $12\,$GHz and confirm the mode is evanescent below cutoff
(with decay rate $\alpha$) and propagating above ({numref}`fig-wg-dispersion`). The
animation ({numref}`fig-wg-propagation`) shows the dominant mode travelling down the
guide above cutoff, the wave the whole device exists to carry.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.check(
    arg_below < 0 < arg_above,
    "below cutoff the mode is evanescent (β imaginary); above cutoff it propagates",
)

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


## Exercise 4 — The eigenvalue problem, numerically

Here the threads converge. The transverse Helmholtz problem {eq}`eq-eigenproblem` is an
eigenvalue problem for the boundary-constrained Laplacian: discretize $-\nabla_T^2$ on
the cross-section with the [§3.4](laplace-poisson.ipynb) five-point stencil and
Dirichlet walls, and its eigenvalues are the squared cutoffs $k_c^2$, its eigenvectors
the TM mode patterns. This is the [§3.4](laplace-poisson.ipynb) boundary-value problem
solved as the [§0.5](../00-foundations/eigenvalues-svd.ipynb) eigenproblem, giving the
[§2.7](../02-classical-mechanics/small-oscillations.ipynb) normal modes, now for fields.

**This exercise (worked).**

1. Build the sparse operator with the `neg_laplacian` helper on an
   $N_x\times N_y$ interior grid and solve for the lowest eigenvalues with
   `scipy.sparse.linalg.eigsh` (shift-invert about zero for the smallest).
2. Compare the numerical $k_c^2$ to the analytic TM cutoffs for the **low**
   modes ({numref}`fig-wg-eigenmode`); the higher modes drift downward because
   the five-point Dirichlet Laplacian under-estimates eigenvalues and the fixed
   grid under-resolves their finer structure — a resolution effect, not an
   error in the physics.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    kc2_num[:3],
    kc2_analytic[:3],
    "the sparse eigensolver recovers the analytic TM cutoffs for the low modes",
    rtol=2e-2,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 5 — A non-separable cross-section

The analytic sine products work only because the rectangle separates. Real guides come
in shapes that do not, and there the numerical eigensolver earns its keep. Consider an
**L-shaped** cross-section, a square with one quadrant removed
({numref}`fig-wg-lshape`): there is no closed-form mode, but the same sparse Dirichlet
Laplacian gives its cutoffs and patterns directly.

**This exercise (student).**

1. Build the L-shaped mask on the grid, assemble `neg_laplacian` over it,
   and solve with `scipy.sparse.linalg.eigsh`.
2. With no closed form to check against, validate by self-consistency: every
   eigenpair satisfies the discretized Helmholtz problem $A\psi=k_c^2\psi$
   to machine precision, and the lowest cutoff is real and positive.

The fundamental mode concentrates in the body of the L, avoiding the
re-entrant corner.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.check(
    residual < 1e-6 and kc2_L.min() > 0,
    "the numerical eigensolver gives self-consistent modes of the non-separable L-guide",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 6 — Cavity resonances

Cap the guide at both ends and the wave is trapped in all three directions. Only
standing waves that fit the box survive, at the discrete resonant frequencies
$f_{mnp}=\tfrac{c}{2}\sqrt{(m/a)^2+(n/b)^2+(p/d)^2}$ {eq}`eq-cavity`
({numref}`fig-wg-cavity`). A microwave cavity is exactly this: a resonator that rings
only at its eigenfrequencies, the electromagnetic cousin of an organ pipe.

**This exercise (worked).** For a cavity $a\times b\times d$ with $a=22.9\,$mm,
$b=10.2\,$mm, $d=30\,$mm, compute the low resonances and confirm the lowest matches the
box formula. These discrete frequencies from 3-D confinement are the direct classical
ancestor of the quantized energies of a quantum particle in a box (Vol VI), the very
same eigenproblem.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    f101,
    0.5 * C_LIGHT * np.sqrt((1 / a_c) ** 2 + (1 / d_c) ** 2),
    "the cavity resonance f_101 matches the box formula",
    rtol=1e-6,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 7 — The drumhead parallel

Nothing in Exercise 4 was about electromagnetism. The transverse Helmholtz eigenproblem
$-\nabla_T^2\psi=k_c^2\psi$ on a box, with $\psi=0$ on the boundary, is *identically* the
equation for the small vibrations of a clamped rectangular **membrane**, a drumhead,
whose eigenvalues are its squared vibration frequencies (up to the wave speed) and whose
eigenvectors are its mode shapes. The waveguide and the drum are the same problem
(callback to the normal modes of
[§2.7](../02-classical-mechanics/small-oscillations.ipynb)).

**This exercise (student).** Solve the 2-D box Helmholtz eigenproblem with the **same**
sparse eigensolver as Exercise 4 and read it as a vibrating membrane: confirm its
eigenvalues are the rectangular-guide cutoffs $k_c^2=(m\pi/a)^2+(n\pi/b)^2$. The match
is exact in form: confinement gives discrete modes whatever the field.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    eig_mem[:3],
    eig_analytic[:3],
    "the drumhead modes are the same eigenproblem as the waveguide TM modes",
    rtol=2e-2,
)

## Exercise 8 — Confinement quantizes

Step back from the machinery to the idea it keeps returning. A wave in open space is
free: any frequency, any direction. The instant we wall it in, that freedom collapses
into a **discrete spectrum**, a countable list of modes each with a cutoff, or, for a
fully closed cavity, a countable list of resonant frequencies. The geometry alone
selects which waves may exist. We met the selection three ways here and they were one
thing: the boundary-value problem of [§3.4](laplace-poisson.ipynb), the eigenvalue
problem of [§0.5](../00-foundations/eigenvalues-svd.ipynb), and the normal modes of
[§2.7](../02-classical-mechanics/small-oscillations.ipynb), all the statement
$-\nabla^2\psi=k^2\psi$ with $\psi$ vanishing on the
walls.

That is the classical rehearsal for the central fact of the quantum world. Replace the
electromagnetic wave by a matter wave and the conducting box by a potential well, and
the identical eigenproblem returns as the **quantized energies of a particle in a box**
(Vol VI). The discreteness that seems so strange in quantum mechanics is, in its
mathematics, nothing more exotic than a wave in a box, which we can see and compute
here.

**This exercise.** Make the unification concrete one last time, with two
measured identities:

1. The lowest membrane eigenvalue of Exercise 7 equals its analytic $k^2$ —
   the drumhead *is* the waveguide eigenproblem.
2. The cavity resonance of Exercise 6 decomposes into guide physics:
   $f_{101}^2 = f_c(\mathrm{TE}_{10})^2 + (c/2d)^2$, the transverse cutoff
   of Exercise 2 plus a longitudinal half-wave — a 3-D resonance is a guided
   mode standing between the end walls.

In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.check(
    same_eigenproblem and cavity_rel < 1e-6,
    "confinement quantizes: the drumhead matches the guide eigenproblem, and the cavity "
    "resonance decomposes into guide cutoff + longitudinal half-wave",
)

## Notebook summary

- A hollow conductor turns the wave equation into a transverse **Helmholtz eigenvalue
  problem** {eq}`eq-eigenproblem`; the rectangular guide's TM modes are sine products
  $\sin(m\pi x/a)\sin(n\pi y/b)$ with $k_c^2=(m\pi/a)^2+(n\pi/b)^2$, verified to satisfy
  $\nabla_T^2\psi=-k_c^2\psi$.
- **Cutoffs** $f_c=c\,k_c/2\pi$: the dominant WR-90 mode is TE$_{10}$ at $c/2a=6.55\,$GHz,
  setting the single-mode band up to TE$_{20}$ at $13.1\,$GHz. Below cutoff a mode is
  **evanescent** ($\alpha=55\,$/m at $6\,$GHz); above it propagates ($\beta=96\,$/m at
  $8\,$GHz), with the dominant mode animated travelling down the guide.
- A **sparse Dirichlet Laplacian** + `scipy.sparse.linalg.eigsh` recovers the analytic
  cutoffs for the low modes (to a couple of percent on this grid, the high modes drifting
  as resolution falls behind their structure), and it cracks a **non-separable** L-shaped
  guide that no sine product can.
- Closing the guide gives a 3-D **cavity** with discrete resonances $f_{mnp}=\tfrac{c}{2}
  \sqrt{(m/a)^2+(n/b)^2+(p/d)^2}$; the **drumhead** carries the identical eigenproblem.
  Confinement quantizes, the classical rehearsal for the quantum particle in a box.

## Outlook

- **Other guides.** Coaxial lines, dielectric and optical-fibre guides, and the quality
  factor $Q$ that measures how sharply a cavity resonates.
- **Dispersion and group velocity.** A guided mode's $\beta(\omega)$ makes the guide
  dispersive, so signals spread; the group velocity $d\omega/d\beta$ governs how fast
  information actually travels.
- **Cavity QED (Vol VI).** The cavity is the classical ancestor not only of the quantum
  box but of cavity quantum electrodynamics, where a single mode couples to a single atom.
- **Arbitrary cross-sections.** The finite-difference eigensolver here generalises to the
  finite-element method for any shape, the working tool of microwave design.
- **Forward links.** Radiation, how fields escape a structure rather than ride it
  ([§3.10](radiation.ipynb));
  and the quantum particle in a box (Vol VI), the same eigenproblem with a matter wave.

In [ ]:
from ecp.style import footer

footer()